# Generative Recommendation System — Analysis

MovieLens-1M · Dot Product Baseline vs LLM-Augmented Pipeline

**Pipeline:** user watch history → LLM taste profile → FAISS semantic retrieval → top-10 recommendations  
**Model:** Llama-3.1-8B-Instant via Groq  
**Embeddings:** `sentence-transformers/all-MiniLM-L6-v2`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Load Dataset

In [ ]:
from src.data_pipeline import load_dataset

train_history, test_items, item_metadata, all_item_ids = load_dataset()
print(f"Users:  {len(train_history):,}")
print(f"Items:  {len(all_item_ids):,}")
print(f"Interactions (train): {sum(len(v) for v in train_history.values()):,}")

## 2. Dataset Overview

In [ ]:
history_lengths = [len(v) for v in train_history.values()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# History length distribution
axes[0].hist(history_lengths, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Training history length (# movies)')
axes[0].set_ylabel('# users')
axes[0].set_title('User History Length Distribution')
axes[0].axvline(np.median(history_lengths), color='tomato', linestyle='--', label=f'Median: {np.median(history_lengths):.0f}')
axes[0].legend()

# Genre distribution
from collections import Counter
genre_counts = Counter()
for meta in item_metadata.values():
    for g in meta['genres'].split(', '):
        genre_counts[g] += 1

genres, counts = zip(*genre_counts.most_common(12))
axes[1].barh(genres[::-1], counts[::-1], color='steelblue')
axes[1].set_xlabel('# movies')
axes[1].set_title('Top Genres in Catalog')

plt.tight_layout()
plt.show()
print(f"Median history length: {np.median(history_lengths):.0f} movies")
print(f"Mean history length:   {np.mean(history_lengths):.1f} movies")

## 3. Evaluation Results

Full evaluation on all 6,034 users. Generative system uses LLM verbalization + FAISS semantic retrieval (reranking disabled by default).

In [ ]:
results = {
    'Dot Product Baseline': {
        'NDCG@10': 0.0044, 'Precision@10': 0.0009, 'HitRate@10': 0.0093, 'Coverage': 0.2341
    },
    'Generative (Groq/Llama)': {
        'NDCG@10': 0.0155, 'Precision@10': 0.0029, 'HitRate@10': 0.0288, 'Coverage': 0.4437
    },
}

df = pd.DataFrame(results).T
df['NDCG@10 lift'] = df['NDCG@10'].pct_change().fillna(0) * 100
df['HitRate lift'] = df['HitRate@10'].pct_change().fillna(0) * 100
df['Coverage lift'] = df['Coverage'].pct_change().fillna(0) * 100
df.round(4)

In [ ]:
metrics = ['NDCG@10', 'Precision@10', 'HitRate@10', 'Coverage']
labels = ['NDCG@10', 'Precision@10', 'HitRate@10', 'Coverage']
baseline_vals = [results['Dot Product Baseline'][m] for m in metrics]
gen_vals = [results['Generative (Groq/Llama)'][m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, baseline_vals, width, label='Dot Product Baseline', color='#7fb3d3')
bars2 = ax.bar(x + width/2, gen_vals, width, label='Generative (Groq/Llama)', color='#2e86ab')

for bar, val in zip(bars1, baseline_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005, f'{val:.4f}',
            ha='center', va='bottom', fontsize=8, color='gray')
for bar, val in zip(bars2, gen_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005, f'{val:.4f}',
            ha='center', va='bottom', fontsize=8, color='#2e86ab', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Score')
ax.set_title('Baseline vs Generative System — MovieLens-1M (6,034 users)', fontsize=13)
ax.legend()

# Lift annotations
for i, (b, g) in enumerate(zip(baseline_vals, gen_vals)):
    lift = (g - b) / b * 100
    ax.annotate(f'+{lift:.0f}%', xy=(x[i] + width/2, g + 0.001),
                fontsize=9, color='green', ha='center')

plt.tight_layout()
plt.show()

## 4. Example: User Profile Verbalization

Pick a user and see how the LLM describes their taste.

In [ ]:
import os
from groq import Groq
from src.verbalizer import verbalize_user_profile

api_key = os.environ.get('GROQ_API_KEY', '')
if not api_key:
    print("Set GROQ_API_KEY to run this cell.")
else:
    client = Groq(api_key=api_key)

    # Pick a user with a long history for a richer profile
    sample_user = max(train_history, key=lambda u: len(train_history[u]))
    history = train_history[sample_user]

    print(f"User {sample_user} — {len(history)} movies watched")
    print("\nLast 10 watched:")
    for iid in history[-10:]:
        m = item_metadata.get(iid, {})
        print(f"  - {m.get('title', iid)} ({m.get('genres', '')})")

    print("\nLLM-generated taste profile:")
    profile = verbalize_user_profile(history, item_metadata, client)
    print(f"  '{profile}'")

## 5. Example: Side-by-Side Recommendations

In [ ]:
if not api_key:
    print("Set GROQ_API_KEY to run this cell.")
else:
    from src.embeddings import build_item_embeddings, build_faiss_index, embed_text, get_model, retrieve_candidates
    from src.baseline import get_user_embedding, recommend_dot_product

    embed_model = get_model()
    item_embeddings = build_item_embeddings(item_metadata, all_item_ids, embed_model)
    faiss_index, id_to_idx, idx_to_id = build_faiss_index(item_embeddings, all_item_ids)

    exclude = set(history)

    # Baseline
    user_vec = get_user_embedding(history, item_embeddings, id_to_idx)
    baseline_recs = recommend_dot_product(user_vec, item_embeddings, all_item_ids, top_k=10, exclude_ids=exclude)

    # Generative
    profile_vec = embed_text(profile, embed_model)
    gen_recs = retrieve_candidates(profile_vec, faiss_index, idx_to_id, top_k=10, exclude_ids=exclude)

    print(f"Profile: '{profile}'\n")
    print(f"{'Rank':<5} {'Dot Product Baseline':<40} {'Generative System':<40}")
    print('-' * 85)
    for i, (b, g) in enumerate(zip(baseline_recs, gen_recs), 1):
        bm = item_metadata.get(b, {})
        gm = item_metadata.get(g, {})
        print(f"{i:<5} {bm.get('title','?')[:38]:<40} {gm.get('title','?')[:38]:<40}")

## 6. Coverage Deep Dive

Coverage measures what fraction of the 3,883-item catalog ever appears in any recommendation. Higher coverage = more diverse, less popularity-biased.

In [ ]:
coverage_data = {
    'System': ['Dot Product Baseline', 'Generative (Groq/Llama)'],
    'Items recommended': [int(0.2341 * len(all_item_ids)), int(0.4437 * len(all_item_ids))],
    'Items never recommended': [len(all_item_ids) - int(0.2341 * len(all_item_ids)),
                                 len(all_item_ids) - int(0.4437 * len(all_item_ids))],
}

fig, ax = plt.subplots(figsize=(9, 3.5))
systems = coverage_data['System']
recommended = coverage_data['Items recommended']
not_recommended = coverage_data['Items never recommended']

ax.barh(systems, recommended, color='#2e86ab', label='Recommended')
ax.barh(systems, not_recommended, left=recommended, color='#d3d3d3', label='Never recommended')

for i, (r, n) in enumerate(zip(recommended, not_recommended)):
    pct = r / (r + n) * 100
    ax.text(r / 2, i, f'{r:,} items ({pct:.1f}%)', va='center', ha='center', color='white', fontweight='bold')

ax.set_xlabel('# items in 3,883-item catalog')
ax.set_title('Catalog Coverage: how many items ever get recommended?')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()